# 04 Baselines and paired statistics

**Purpose.** Compare the seed-42 hybrid model with nine classical comparators on the same 7,500 test flows: paired accuracy difference with a paired-bootstrap 95% interval, odds ratio of discordant pairs, McNemar test with Holm adjustment; and Hedges' g for the five-seed parameter-matched controls.

**Method notes.** Variation over test flows is handled by paired tests on the same flows; variation over training runs by mean, standard deviation and Hedges' g. A five-pair Wilcoxon test is not used as evidence (its minimum two-sided p is 0.0625). Absence of a significant difference is not proof of equivalence.

**Inputs.** `inputs/baseline_predictions/*/predictions.npz`, `archive/*_predictions.npz`. **Outputs.** `results/ci_effect_sizes.json`.

In [1]:
# --- setup: work from the laptop/ folder so that relative paths resolve ---
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
pd.set_option("display.precision", 4)
print("working folder:", ROOT)

working folder: <repo>\local_analysis


In [2]:
import subprocess
r = subprocess.run([sys.executable, "ci_effect_sizes.py"], capture_output=True, text=True)
print("exit code", r.returncode, r.stderr[-300:])
res = json.load(open("results/ci_effect_sizes.json"))
res["headline_multiclass_seed42"], res["headline_binary_seed42"]

exit code 0 


({'acc': 0.7918666666666667,
  'acc_ci95': [0.7825333333333333, 0.8010666666666667],
  'f1w': 0.7708048050063878,
  'f1w_ci95_1000boot': [0.7610463954144968, 0.7817331422516018]},
 {'acc': 0.9962666666666666, 'acc_ci95': [0.9948, 0.9976]})

In [3]:
from scipy.stats import chi2
paired = res["paired_vs_baselines_multiclass_seed42"]
rows = []
for k, v in paired.items():
    b_, c_ = v["discordant_ours_only"], v["discordant_baseline_only"]
    rows.append({"comparator": k, "comparator acc": v["acc_baseline"], "diff (pp)": v["diff_pp"],
                 "CI low": v["diff_pp_ci95"][0], "CI high": v["diff_pp_ci95"][1], "odds ratio b/c": v["odds_ratio"],
                 "p (McNemar)": float(chi2.sf((abs(b_ - c_) - 1) ** 2 / (b_ + c_), 1))})
t = pd.DataFrame(rows).sort_values("comparator acc", ascending=False).reset_index(drop=True)
order = t["p (McNemar)"].argsort().values; m_ = len(t); run = 0; holm = np.empty(m_)
for rank, i in enumerate(order):
    run = max(run, min(1.0, t.loc[i, "p (McNemar)"] * (m_ - rank))); holm[i] = run
t["p (Holm)"] = holm
t

,comparator,comparator acc,diff (pp),CI low,CI high,odds ratio b/c,p (McNemar),p (Holm)
0,Random_Forest,0.8275,-3.5600,-4.3733,-2.7467,0.5782,3.9000e-17,3.5100e-16
1,Gradient_Boosting,0.8209,-2.9067,-3.6800,-2.1200,0.6086,4.1843e-13,3.3475e-12
2,KNN,0.8101,-1.8267,-2.6800,-0.9733,0.7769,3.8312e-05,2.2987e-04
3,rf20,0.8009,-0.9067,-1.6133,-0.1867,0.8313,1.3652e-02,4.0955e-02
4,gbdt_shallow,0.7968,-0.4933,-1.2667,0.2933,0.9201,2.2728e-01,2.3299e-01
5,MLP,0.7964,-0.4533,-1.0000,0.0933,0.8571,1.1650e-01,2.3299e-01
6,pruned_mlp,0.7824,0.9467,0.3333,1.5470,1.2934,2.9651e-03,1.1860e-02
7,int8_mlp,0.7815,1.0400,0.4267,1.6403,1.3291,1.0479e-03,5.2395e-03
8,SVM_(RBF),0.7716,2.0267,1.3067,2.7333,1.4872,5.9405e-08,4.1583e-07


In [4]:
pd.DataFrame(res["matched_controls_vs_pqc_hedges_g"]).T

,diff_pp_vs_pqc,hedges_g,g_ci95
Random Fourier map,0.74,1.35,"[-0.02, 2.72]"
Tiny MLP,0.58,1.02,"[-0.3, 2.34]"
Dense linear map,0.45,0.66,"[-0.62, 1.93]"
Identity (no bottleneck),0.18,0.27,"[-0.98, 1.52]"
